# DMF ( Data Metric Function)

Basics

DMF = a function that measures data quality on a table or view column.

Two types: 
- system DMFs (built-in)  
- user-defined DMFs.

Features:
- Results are stored in a dedicated event table.
- Results are viewed via SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS.

System DMFs

- Location: SNOWFLAKE.CORE schema.
- Can’t be modified or renamed.
- Accuracy: NULL_COUNT, NULL_PERCENT, BLANK_COUNT, BLANK_PERCENT
- Freshness: FRESHNESS, DATA_METRIC_SCHEDULED_TIME
- Statistics: AVG, MAX, MIN, STDDEV
- Uniqueness: DUPLICATE_COUNT
- Also: ROW_COUNT, ACCEPTED_VALUES (newer ones keep getting added)

Custom DMFs

- Created with CREATE DATA METRIC FUNCTION.
- Needs the CREATE DATA METRIC FUNCTION privilege on the schema.

Privileges

- Table owner role needs the account-level EXECUTE DATA METRIC FUNCTION.
- To use system DMFs: the CORE-schema database role, or IMPORTED PRIVILEGES on SNOWFLAKE.

Attach and manage

- Add: ALTER TABLE/VIEW ... ADD DATA METRIC FUNCTION <dmf> ON (col)
- Drop: ALTER TABLE/VIEW ... DROP DATA METRIC FUNCTION
- Suspend or resume: MODIFY DATA METRIC FUNCTION (RESUME parameter)
Or use the WITH DATA METRIC FUNCTION clause in CREATE.

Scheduling

- Parameter: DATA_METRIC_SCHEDULE (set at table or view level).
- One schedule is shared by all DMFs on that object.
- Options: '5 MINUTE', USING CRON ..., 'TRIGGER_ON_CHANGES'.
- Empty string '' disables the schedule.
- if the previous run is still going, the next run is skipped.

Cost

- Scheduled runs: billed (serverless compute).
- Manual SELECT call: not billed as DMF usage.

Expectations

- Pass/fail rule on a DMF result, added with ADD/MODIFY EXPECTATION.
- Names must be unique within one DMF binding.


# Questions 

Q1. A team attaches NULL_COUNT and DUPLICATE_COUNT to one table. They want NULL_COUNT every 5 minutes and DUPLICATE_COUNT once a day. What is the best answer?

- A. Set a separate schedule on each DMF
- B. Set two DATA_METRIC_SCHEDULE values on the table
- C. This isn’t possible on one table, because all DMFs on a table share one schedule
- D. Use TRIGGER_ON_CHANGES for one and cron for the other

Answer: C. One table, one schedule. (If they really need two rhythms, they would have to rethink the setup, like using a view.)

Q2. An engineer runs ALTER TABLE ... ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON (email) and it fails with a privilege error. The engineer has plenty of table privileges. What is the most likely missing piece?

- A. USAGE on the warehouse
- B. The table owner role lacks the account-level EXECUTE DATA METRIC FUNCTION privilege
- C. CREATE DATA METRIC FUNCTION on the schema
- D. ACCOUNTADMIN role is mandatory

Answer: B. The owner role of the table needs that account-level privilege. Option C is for creating custom DMFs, not attaching.

Q3. A developer is testing and runs SELECT SNOWFLAKE.CORE.NULL_COUNT(SELECT ssn FROM customers); fifty times in a day. What does this do to the DMF scheduled billing?

- A. Each call is billed as serverless DMF usage
- B. Calls are billed only if the schedule is active
- C. It is not billed as scheduled DMF usage
- D. Calls are blocked unless a schedule exists

Answer: C. Manual calls are not billed as scheduled DMF usage. Test as much as you like (just remember your warehouse still runs the query).

Q4. A DMF is scheduled with cron to run every 10 minutes, but a run sometimes takes 15 minutes. What happens when the next valid time arrives while the previous run is still going?

- A. Both runs execute in parallel
- B. The new run is queued
- C. The new run is skipped
- D. The earlier run is cancelled

Answer: C. If the previous run hasn’t finished, that valid run time is skipped.

Q5. A data steward wants to see the history of DMF results for table CUSTOMERS. Where should they look?

- A. INFORMATION_SCHEMA.TABLES
- B. SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
- C. SNOWFLAKE.CORE.RESULTS
- D. SHOW PARAMETERS

Answer: B. Results are recorded in the DMF event table and read through that view.

Q6. A table is loaded unpredictably by many pipelines. The team wants DMFs to run when the data changes, not on a clock. Which setting fits best?

- A. DATA_METRIC_SCHEDULE = '1 MINUTE'
- B. DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES'
- C. DATA_METRIC_SCHEDULE = ''
- D. USING CRON * * * * *

Answer: B. TRIGGER_ON_CHANGES is made for this. Option C actually turns the schedule off.

Q7. A manager asks you to change SNOWFLAKE.CORE.NULL_COUNT so it ignores a special value “N/A”. What is the right approach?

- A. ALTER FUNCTION on the system DMF
- B. Edit the function in the CORE schema
- C. System DMFs can’t be changed, so create a custom DMF with your own logic
- D. Ask Snowflake Support to patch your account’s copy

Answer: C. System DMFs are maintained by Snowflake and fixed. Custom DMF is the way.

Q8. A role must create a custom DMF in schema GOVERNANCE.DMFS. Which privilege is needed? (Select the best answer.)

- A. EXECUTE DATA METRIC FUNCTION on the account
- B. CREATE DATA METRIC FUNCTION on the schema
- C. CREATE FUNCTION on the account
- D. OWNERSHIP on the database

Answer: B. Creating needs CREATE DATA METRIC FUNCTION on the schema. Keep A for attaching.

Q9. A team wants to pause just one DMF on a table for a week, without touching the others, and resume it later. Which is correct?

- A. Set the table schedule to ''
- B. Drop all DMFs and re-add them
- C. Use ALTER TABLE ... MODIFY DATA METRIC FUNCTION ... to suspend it, then another MODIFY with RESUME later
- D. Revoke EXECUTE DATA METRIC FUNCTION from the owner

Answer: C. MODIFY handles one DMF. Option A would stop all of them.

Q10. The finance team wants data quality checks on a view, not the base table. Is this supported?

- A. No, DMFs work only on tables
- B. Yes, with ALTER VIEW ... ADD DATA METRIC FUNCTION
- C. Yes, but only with custom DMFs
- D. Only on materialized views

Answer: B. The docs show ALTER VIEW with a DMF (for example ROW_COUNT), so views are supported.